# 🧠 AI CBT Conversation Dataset Pipeline

## 🎯 Project Overview

This project aims to build a high-quality conversational dataset for training an AI assistant capable of providing **empathetic and supportive responses**, inspired by Cognitive Behavioral Therapy (CBT) principles.

The goal is to:
- Aggregate multiple conversational datasets
- Clean and standardize them
- Transform them into a unified format
- Prepare them for **LLM fine-tuning**

---

## 📦 Datasets Used

- **DailyDialog** → General human conversations
- **SkillTalk (Blended Skill Talk)** → Guided conversational responses
- **GoEmotions** → Emotion classification (used later for safety/monitoring)

---

## ⚙️ Pipeline Steps

1. Data Loading  
2. Exploratory Data Analysis (EDA)  
3. Data Cleaning & Unification  
4. LLM Formatting (Chat format)  
5. (Next) Fine-tuning preparation  

---

## 🧠 Final Objective

Produce a dataset in **chat format** compatible with models like:
- Mistral
- LLaMA
- Hugging Face Transformers

---

## 🚀 Outcome

A production-ready dataset:
- Clean
- Structured
- Ready for fine-tuning

# 📂 1. Data Loading

## 🎯 Objective

Load multiple conversational datasets from different formats (TXT, Parquet) and prepare them for analysis.

---

## 🧠 Why this step matters

Real-world ML projects rarely use a single clean dataset.  
This step ensures we can handle:
- Multiple data formats
- Different schemas
- Raw and unstructured data

---

## 📌 Expected Output

- DataFrames for each dataset
- Basic understanding of structure and columns

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# Load raw dialogues
with open("/content/dialogues_train.txt", "r") as f:
    dialogs = f.readlines()

print(f"Number of dialogues: {len(dialogs)}")
print(dialogs[0])

In [ ]:
pairs = []

for dialog in dialogs:
    sentences = dialog.split("__eou__")
    sentences = [s.strip() for s in sentences if s.strip()]

    for i in range(len(sentences) - 1):
        pairs.append({
            "input": sentences[i],
            "output": sentences[i+1]
        })

df_daily = pd.DataFrame(pairs)

print(df_daily.shape)
df_daily.head()

In [ ]:
# GoEmotions
df_go = pd.read_parquet("/content/train-goemotions-00000-of-00001.parquet")

# SkillTalk
df_bst = pd.read_parquet("/content/train-skilltalk-00000-of-00001.parquet")

print("GoEmotions:", df_go.shape)
print("SkillTalk:", df_bst.shape)

In [ ]:
print("DailyDialog columns:", df_daily.columns)
print("GoEmotions columns:", df_go.columns)
print("SkillTalk columns:", df_bst.columns)

In [ ]:
df_daily.sample(3)

In [ ]:
df_bst.sample(3)

# 📊 2. Data Understanding & EDA

## 🎯 Objective

Explore the datasets to understand:
- Data distribution
- Text length
- Data quality
- Potential issues

---

## 🔍 Key Analysis

- Missing values
- Text length distribution
- Sample inspection

---

## 🧠 Why this step matters

EDA helps identify:
- Noisy data
- Outliers
- Formatting inconsistencies

This step directly informs the **cleaning strategy**.

---

## 📌 Observations

- Most responses are short conversational exchanges
- Some responses are too long or too short
- Data appears mostly clean but requires normalization

In [ ]:
print("DailyDialog shape:", df_daily.shape)
print("GoEmotions shape:", df_go.shape)
print("SkillTalk shape:", df_bst.shape)

In [ ]:
# Missing values

print("DailyDialog nulls:\n", df_daily.isnull().sum())
print("\nGoEmotions nulls:\n", df_go.isnull().sum())
print("\nSkillTalk nulls:\n", df_bst.isnull().sum())

In [ ]:
# Text Length Analysis

df_daily["input_length"] = df_daily["input"].apply(lambda x: len(str(x).split()))
df_daily["output_length"] = df_daily["output"].apply(lambda x: len(str(x).split()))

df_daily[["input_length", "output_length"]].describe()

In [ ]:
# Visualisation

df_daily["input_length"] = df_daily["input"].apply(lambda x: len(str(x).split()))
df_daily["output_length"] = df_daily["output"].apply(lambda x: len(str(x).split()))

df_daily[["input_length", "output_length"]].describe()

In [ ]:
df_daily["output_length"].hist(bins=50)
plt.title("Output Length Distribution")
plt.show()

# 🧹 3. Data Cleaning & Format Unification

## 🎯 Objective

Transform multiple datasets into a **single unified format**:

input → output

---

## 🔧 Cleaning Steps

- Text normalization (spaces, formatting)
- Removal of short/empty samples
- Conversion of structured fields into text
- Duplicate removal
- Length filtering

---

## 🔄 Dataset Transformation

- DailyDialog → already conversational
- SkillTalk → converted into input/output format
- GoEmotions → excluded (used later for classification)

---

## 🧠 Why this step matters

Models are extremely sensitive to data quality.

Bad data → bad model  
Clean data → strong model

---

## 📌 Final Result

A clean dataset with:
- Consistent structure
- High-quality conversational pairs
- Ready for LLM formatting

In [ ]:
# Cleaning function

def clean_text(x):
    x = str(x)
    x = x.strip()
    x = x.replace("\n", " ")
    x = x.replace("\t", " ")
    x = " ".join(x.split())
    return x

def clean_skilltalk_output(x):
    x = str(x).strip()

    # Remove outer brackets if present
    if x.startswith("[") and x.endswith("]"):
        x = x[1:-1]

    x = x.replace("\n", " ")
    x = x.replace("\t", " ")
    x = " ".join(x.split())
    return x

In [ ]:
#  Clean DailyDialog

df_daily = df_daily[["input", "output"]].copy()

df_daily["input"] = df_daily["input"].apply(clean_text)
df_daily["output"] = df_daily["output"].apply(clean_text)

df_daily = df_daily[
    (df_daily["input"].str.len() > 5) &
    (df_daily["output"].str.len() > 5)
]

print("DailyDialog:", df_daily.shape)
df_daily.head()

In [ ]:
df_bst_clean = df_bst[["previous_utterance", "guided_messages"]].copy()

df_bst_clean = df_bst_clean.rename(columns={
    "previous_utterance": "input",
    "guided_messages": "output"
})

In [ ]:
df_bst_clean["input"] = df_bst_clean["input"].apply(clean_text)
df_bst_clean["output"] = df_bst_clean["output"].apply(clean_skilltalk_output)

In [ ]:
df_bst_clean = df_bst_clean[
    (df_bst_clean["input"].str.len() > 5) &
    (df_bst_clean["output"].str.len() > 5)
]

print("SkillTalk:", df_bst_clean.shape)
df_bst_clean.head()

In [ ]:
df_final = pd.concat([df_daily, df_bst_clean], ignore_index=True)

print("Merged:", df_final.shape)
df_final.sample(10)

In [ ]:
df_final = df_final.drop_duplicates()

print("After dropping duplicates:", df_final.shape)

In [ ]:
df_final["input_length"] = df_final["input"].apply(lambda x: len(x.split()))
df_final["output_length"] = df_final["output"].apply(lambda x: len(x.split()))

In [ ]:
df_final = df_final[
    (df_final["input_length"] >= 2) &
    (df_final["output_length"] >= 2) &
    (df_final["input_length"] <= 80) &
    (df_final["output_length"] <= 80)
]

print("After length filtering:", df_final.shape)
df_final.sample(10)

In [ ]:
print(df_final.isnull().sum())
print(df_final[["input_length", "output_length"]].describe())

In [ ]:
df_final.to_csv("/content/final_conversation_dataset.csv", index=False)

# 🤖 4. LLM Dataset Formatting

## 🎯 Objective

Convert the cleaned dataset into a format suitable for **LLM fine-tuning**.

---

## 🧠 Target Format

We use a **chat-based structure**:

{
  "messages": [
    {"role": "system": "..."},
    {"role": "user": "..."},
    {"role": "assistant": "..."}
  ]
}

---

## 🔄 Transformation

Each row:
- input → user message
- output → assistant response

---

## 🧠 Why this format?

This is the standard format used by:
- LLaMA
- Mistral
- OpenAI-style chat models

---

## 📦 Output

- JSON dataset ready for fine-tuning
- Compatible with Hugging Face training pipelines

---

## 🚀 Final Result

A production-ready dataset for training an AI conversational assistant.

In [ ]:
df_final = df_final[["input", "output"]].copy()

print(df_final.shape)
df_final.head()

In [ ]:
def format_chat(row):
    return {
        "messages": [
            {"role": "system", "content": "You are a helpful and empathetic conversation coach."},
            {"role": "user", "content": row["input"]},
            {"role": "assistant", "content": row["output"]}
        ]
    }

chat_data = df_final.apply(format_chat, axis=1).tolist()

print("Number of samples:", len(chat_data))
chat_data[:2]

In [ ]:
import json

with open("llm_chat_dataset.json", "w", encoding="utf-8") as f:
    json.dump(chat_data, f, ensure_ascii=False, indent=2)

print("JSON saved successfully")

In [ ]:
with open("llm_chat_dataset.json", "r", encoding="utf-8") as f:
    data_check = json.load(f)

print("Saved records:", len(data_check))
print(data_check[0])

In [ ]:
from google.colab import files

files.download("llm_chat_dataset.json")

In [ ]:
df_final.to_csv("final_dataset.csv", index=False)
files.download("final_dataset.csv")

# ✅ Conclusion

## 🎯 What was achieved

- Loaded multiple real-world datasets
- Performed exploratory data analysis
- Built a full cleaning pipeline
- Unified heterogeneous data sources
- Generated a chat-formatted dataset for LLM training

---

## 🧠 Key Takeaways

- Data quality is critical in NLP tasks
- Real datasets require significant preprocessing
- Structuring data properly is essential for model performance

---

## 🚀 Next Steps

- Fine-tune an LLM (LoRA / Hugging Face)
- Add safety classifier (emotion detection)
- Deploy as an API for real-world usage

---

## 💡 Project Value

This project demonstrates:
- Data engineering for ML
- NLP preprocessing skills
- Understanding of LLM training pipelines

---

🔥 This dataset is now ready for real-world AI applications.